 # Phân tích Án lệ: Thống kê vi phạm theo Điều và Chương

 Notebook này xử lý danh sách các file JSON (định dạng `stage2.json`)

 để trích xuất các "Điều" từ trường `QUYET_DINH`. Sau đó ánh xạ

 các "Điều" này sang "Chương" tương ứng thông qua `law_doc.json`

 và hiển thị những Điều, Chương bị vi phạm nhiều nhất (mỗi case chỉ đếm 1 lần cho 1 Chương).

In [1]:
import json
import re
import glob
import os
from pathlib import Path
from collections import Counter

 ## 1. Load dữ liệu ánh xạ Luật

 Đọc `law_doc.json` để tạo từ điển ánh xạ từ "Điều" sang "Chương".

In [2]:
DATA_DIR = Path('data_create/extracted_fields/2024-2025')
CASELAW_FILES_PATTERN = str(DATA_DIR / '*.json')
LAW_DOC_PATH = Path('law_doc.json')


def build_dieu_to_chuong_map(law_doc_path):
    dieu_to_chuong = {}
    try:
        with open(law_doc_path, 'r', encoding='utf-8') as f:
            law_data = json.load(f)

        for chuong in law_data:
            if chuong.get("type") == "Chương":
                chuong_idx = str(chuong.get("index", "Unknown"))
                for dieu in chuong.get("Điều", []):
                    if dieu.get("type") == "Điều":
                        dieu_idx = str(dieu.get("index", "")).strip()
                        if dieu_idx:
                            dieu_to_chuong[dieu_idx] = chuong_idx
    except Exception as e:
        print(f"Lỗi khi load {law_doc_path}: {e}")

    return dieu_to_chuong


dieu_to_chuong = build_dieu_to_chuong_map(LAW_DOC_PATH)
print(f"Đã load thành công ánh xạ cho {len(dieu_to_chuong)} Điều.")
print(f"Thư mục dữ liệu đang dùng: {DATA_DIR}")


Đã load thành công ánh xạ cho 427 Điều.
Thư mục dữ liệu đang dùng: data_create/extracted_fields/2024-2025


## 2. Trích xuất dữ liệu từ schema hiện tại

Các file trong `data_create/extracted_fields/2024-2025/` dùng schema mới gồm `Danh_sach_bi_cao`, `Nhan_dinh_cua_toa_an`, `Noi_dung_vu_an`, `QUYET_DINH`. Các helper bên dưới vẫn có fallback cho schema cũ (`THONG_TIN_CHUNG`, `De_Nghi_Cua_Vien_Kiem_Sat`, `PHAN_QUYET_CUA_TOA_SO_THAM`) để notebook không bị vỡ khi chạy lẫn dữ liệu cũ.


In [3]:
ARTICLE_PATTERN = re.compile(r'(?i)điều\s+(\d+[a-zA-Z]*)')


def extract_dieu_from_text(text):
    if not text:
        return []
    return ARTICLE_PATTERN.findall(str(text))


def is_blhs_reference(rule):
    bo_luat = str(rule.get("Bo_Luat_Va_Van_Ban_Khac", "")).strip().lower()
    return "blhs" in bo_luat or "bộ luật hình sự" in bo_luat


def extract_dieu_from_case(case_data):
    """Extract cited articles from new 2024-2025 files and legacy structured files."""
    extracted_dieus = []

    # Legacy structured schema.
    for list_key in ("PHAN_QUYET_CUA_TOA_SO_THAM", "De_Nghi_Cua_Vien_Kiem_Sat"):
        values = case_data.get(list_key, [])
        for item in values if isinstance(values, list) else []:
            if not isinstance(item, dict):
                continue
            can_cu_list = item.get("Can_Cu_Dieu_Luat", [])
            if not isinstance(can_cu_list, list):
                continue
            for rule in can_cu_list:
                if isinstance(rule, dict) and is_blhs_reference(rule):
                    dieu_idx = str(rule.get("Dieu", "")).strip()
                    if dieu_idx:
                        extracted_dieus.append(dieu_idx)

    if extracted_dieus:
        return extracted_dieus

    # Current 2024-2025 schema stores the legal basis as raw text.
    text_parts = [case_data.get("QUYET_DINH", "")]
    noi_dung = case_data.get("Noi_dung_vu_an", {})
    if isinstance(noi_dung, dict):
        text_parts.extend(str(v) for v in noi_dung.values())
    nhan_dinh = get_nhan_dinh_text(case_data)
    if nhan_dinh:
        text_parts.append(nhan_dinh)

    return extract_dieu_from_text("\n".join(str(part) for part in text_parts if part))


def normalize_name(name):
    return " ".join(str(name).lower().split())


def extract_name_from_bio(text):
    """Best-effort defendant name extraction from the new Danh_sach_bi_cao bio strings."""
    if not isinstance(text, str):
        return ""
    first_clause = re.split(r'[,;\n]', text.strip(), maxsplit=1)[0]
    first_clause = re.sub(r'^[-+\s]*(bị\s+cáo\s+)?', '', first_clause, flags=re.IGNORECASE).strip()
    return first_clause


def get_defendant_names(case_data):
    names_by_source = {}

    # Current 2024-2025 schema.
    danh_sach = case_data.get("Danh_sach_bi_cao", [])
    if isinstance(danh_sach, list):
        names_by_source["Danh_sach_bi_cao"] = {
            normalize_name(extract_name_from_bio(item))
            for item in danh_sach
            if normalize_name(extract_name_from_bio(item))
        }

    # Legacy schema fallbacks.
    thong_tin_chung = case_data.get("THONG_TIN_CHUNG", {})
    if isinstance(thong_tin_chung, dict):
        thong_tin_bi_cao = thong_tin_chung.get("Thong_Tin_Bi_Cao", [])
        if isinstance(thong_tin_bi_cao, list):
            names_by_source["Thong_Tin_Bi_Cao"] = {
                normalize_name(item.get("Ho_Ten", ""))
                for item in thong_tin_bi_cao
                if isinstance(item, dict) and normalize_name(item.get("Ho_Ten", ""))
            }

    for key in ("De_Nghi_Cua_Vien_Kiem_Sat", "PHAN_QUYET_CUA_TOA_SO_THAM"):
        items = case_data.get(key, [])
        if isinstance(items, list):
            names_by_source[key] = {
                normalize_name(item.get("Bi_Cao", ""))
                for item in items
                if isinstance(item, dict) and normalize_name(item.get("Bi_Cao", ""))
            }

    return {source: names for source, names in names_by_source.items() if names}


def get_nhan_dinh_text(case_data):
    nhan_dinh = case_data.get("Nhan_dinh_cua_toa_an", case_data.get("NHAN_DINH_CUA_TOA_AN", {}))
    if isinstance(nhan_dinh, dict):
        return "\n".join(str(v) for v in nhan_dinh.values() if v)
    if isinstance(nhan_dinh, list):
        return "\n".join(str(v) for v in nhan_dinh if v)
    return str(nhan_dinh or "")


 ## 3. Xử lý các file JSON án lệ

 Đếm số lượng Điều và Chương. Mỗi Chương chỉ đếm 1 lần cho mỗi vụ án, cho dù vụ án đó có vi phạm nhiều Điều thuộc cùng Chương đó.

In [4]:
dieu_counter = Counter()
chuong_counter = Counter()
files_without_articles = []

case_files = sorted(
    f for f in glob.glob(CASELAW_FILES_PATTERN)
    if not os.path.basename(f).lower() == 'law_doc.json'
)

print(f"Tìm thấy {len(case_files)} file án lệ để xử lý trong {DATA_DIR}.\n")

for file_path in case_files:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            case_data = json.load(f)

        unique_dieus_in_case = set(extract_dieu_from_case(case_data))
        if not unique_dieus_in_case:
            files_without_articles.append(os.path.basename(file_path))
            continue

        unique_chuongs_in_case = set()
        for dieu in unique_dieus_in_case:
            dieu_counter[dieu] += 1
            unique_chuongs_in_case.add(dieu_to_chuong.get(dieu, "Unknown/Unmapped"))

        for chuong in unique_chuongs_in_case:
            chuong_counter[chuong] += 1

    except json.JSONDecodeError:
        print(f"Lỗi đọc JSON: {os.path.basename(file_path)}")
    except Exception as e:
        print(f"Lỗi khi xử lý {file_path}: {e}")

print(f"Số file không trích xuất được Điều: {len(files_without_articles)}")


Tìm thấy 1291 file án lệ để xử lý trong data_create/extracted_fields/2024-2025.

Số file không trích xuất được Điều: 0


 ## 4. Hiển thị kết quả

In [5]:
TOP_N = 20

print("="*50)
print(f"🔝 TOP {TOP_N} MOST VIOLATED ARTICLES (ĐIỀU)")
print("="*50)
if not dieu_counter:
    print("Không tìm thấy Điều nào.")
else:
    for rank, (dieu, count) in enumerate(dieu_counter.most_common(TOP_N), 1):
        chuong = dieu_to_chuong.get(dieu, "Unknown")
        print(f"{rank}. Điều {dieu:<5} (Chương {chuong:<3}) : {count} cases")

print("\n" + "="*50)
print(f"🔝 TOP {TOP_N} MOST VIOLATED CHAPTERS (CHƯƠNG)")
print("="*50)
if not chuong_counter:
    print("Không tìm thấy Chương nào.")
else:
    for rank, (chuong, count) in enumerate(chuong_counter.most_common(TOP_N), 1):
        print(f"{rank}. Chương {chuong:<15} : {count} cases")

🔝 TOP 20 MOST VIOLATED ARTICLES (ĐIỀU)
1. Điều 51    (Chương VIII) : 1288 cases
2. Điều 106   (Chương XII) : 1069 cases
3. Điều 47    (Chương VII) : 972 cases
4. Điều 136   (Chương XIV) : 905 cases
5. Điều 38    (Chương VI ) : 791 cases
6. Điều 52    (Chương VIII) : 622 cases
7. Điều 23    (Chương IV ) : 552 cases
8. Điều 30    (Chương VI ) : 491 cases
9. Điều 2     (Chương I  ) : 485 cases
10. Điều 6     (Chương II ) : 476 cases
11. Điều 54    (Chương VIII) : 387 cases
12. Điều 331   (Chương XXII) : 352 cases
13. Điều 135   (Chương XIV) : 316 cases
14. Điều 48    (Chương VII) : 306 cases
15. Điều 468   (Chương Unknown) : 293 cases
16. Điều 333   (Chương XXII) : 290 cases
17. Điều 251   (Chương XX ) : 287 cases
18. Điều 123   (Chương XIV) : 263 cases
19. Điều 58    (Chương VIII) : 237 cases
20. Điều 584   (Chương Unknown) : 208 cases

🔝 TOP 20 MOST VIOLATED CHAPTERS (CHƯƠNG)
1. Chương VIII            : 1288 cases
2. Chương XII             : 1100 cases
3. Chương VII             : 1100 c

In [6]:
json_files_to_check = case_files

total_files = 0
mismatch_files = []
single_source_files = 0

print(f"Bắt đầu kiểm tra số lượng bị cáo trong thư mục '{DATA_DIR}'...")

for file_path in json_files_to_check:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        names_by_source = get_defendant_names(data)
        if len(names_by_source) <= 1:
            single_source_files += 1
        else:
            counts = {source: len(names) for source, names in names_by_source.items()}
            if len(set(counts.values())) > 1:
                mismatch_files.append({
                    "file": os.path.basename(file_path),
                    "counts": counts,
                    "names_by_source": {k: sorted(v) for k, v in names_by_source.items()},
                })

        total_files += 1

    except json.JSONDecodeError:
        pass
    except Exception as e:
        print(f"Lỗi khi xử lý {file_path}: {e}")

print("="*70)
print("KẾT QUẢ KIỂM TRA LỆCH SỐ LƯỢNG BỊ CÁO")
print("="*70)

if total_files == 0:
    print("Không tìm thấy file JSON nào hợp lệ trong thư mục để kiểm tra.")
else:
    mismatch_count = len(mismatch_files)
    mismatch_percentage = (mismatch_count / total_files) * 100

    print(f"Tổng số file đã kiểm tra: {total_files}")
    print(f"Số file chỉ có một nguồn tên bị cáo để kiểm tra: {single_source_files}")
    print(f"Số file bị lệch giữa các nguồn hiện có: {mismatch_count}")
    print(f"Tỷ lệ lỗi: {mismatch_percentage:.2f}%\n")

    if mismatch_count > 0:
        print("DANH SÁCH CÁC FILE LỖI:")
        for item in mismatch_files[:50]:
            print(f"\n- {item['file']}")
            print(f"  Số lượng: {item['counts']}")
            print(f"  Tên trích xuất: {item['names_by_source']}")
        if mismatch_count > 50:
            print(f"... và {mismatch_count - 50} file khác.")
    else:
        print("Không có file nào bị lệch số lượng bị cáo giữa các nguồn hiện có.")


Bắt đầu kiểm tra số lượng bị cáo trong thư mục 'data_create/extracted_fields/2024-2025'...
KẾT QUẢ KIỂM TRA LỆCH SỐ LƯỢNG BỊ CÁO
Tổng số file đã kiểm tra: 1291
Số file chỉ có một nguồn tên bị cáo để kiểm tra: 1291
Số file bị lệch giữa các nguồn hiện có: 0
Tỷ lệ lỗi: 0.00%

Không có file nào bị lệch số lượng bị cáo giữa các nguồn hiện có.


In [7]:
empty_name_files = set()
empty_name_details = []


def check_empty_names(data_list, name_key, source_name, file_name):
    if not isinstance(data_list, list):
        return

    for idx, item in enumerate(data_list):
        if isinstance(item, dict):
            name = item.get(name_key)
        else:
            name = extract_name_from_bio(item)

        if name is None or (isinstance(name, str) and not name.strip()):
            empty_name_files.add(file_name)
            empty_name_details.append({
                "file": file_name,
                "source": source_name,
                "index": idx,
                "key": name_key,
                "value_found": name,
            })

print(f"Bắt đầu kiểm tra tên bị cáo trống trong thư mục '{DATA_DIR}'...")

for file_path in json_files_to_check:
    file_name = os.path.basename(file_path)
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        check_empty_names(data.get("Danh_sach_bi_cao", []), None, "Danh_sach_bi_cao", file_name)

        thong_tin_chung = data.get("THONG_TIN_CHUNG", {})
        if isinstance(thong_tin_chung, dict):
            check_empty_names(thong_tin_chung.get("Thong_Tin_Bi_Cao", []), "Ho_Ten", "Thong_Tin_Bi_Cao", file_name)

        check_empty_names(data.get("De_Nghi_Cua_Vien_Kiem_Sat", []), "Bi_Cao", "De_Nghi_Cua_Vien_Kiem_Sat", file_name)
        check_empty_names(data.get("PHAN_QUYET_CUA_TOA_SO_THAM", []), "Bi_Cao", "PHAN_QUYET_CUA_TOA_SO_THAM", file_name)

    except json.JSONDecodeError:
        pass
    except Exception as e:
        print(f"Lỗi khi xử lý {file_path}: {e}")

print("="*70)
print("KẾT QUẢ KIỂM TRA TÊN BỊ CÁO TRỐNG")
print("="*70)

total_files_checked = len(json_files_to_check)
if total_files_checked == 0:
    print("Không tìm thấy file JSON nào hợp lệ trong thư mục để kiểm tra.")
else:
    empty_error_count = len(empty_name_files)
    empty_error_rate = (empty_error_count / total_files_checked) * 100

    print(f"Tổng số file đã kiểm tra: {total_files_checked}")
    print(f"Số file chứa tên bị cáo trống: {empty_error_count}")
    print(f"Tỷ lệ lỗi: {empty_error_rate:.2f}%\n")

    if empty_error_count > 0:
        print("CHI TIẾT CÁC LỖI TÌM THẤY:")
        for detail in empty_name_details[:100]:
            print(f"- File: {detail['file']} | Nguồn: {detail['source']} | Vị trí: {detail['index']} | Giá trị: {detail['value_found']}")
        if len(empty_name_details) > 100:
            print(f"... và {len(empty_name_details) - 100} lỗi khác.")
    else:
        print("Không có bị cáo nào bị thiếu tên trong các file.")


Bắt đầu kiểm tra tên bị cáo trống trong thư mục 'data_create/extracted_fields/2024-2025'...
KẾT QUẢ KIỂM TRA TÊN BỊ CÁO TRỐNG
Tổng số file đã kiểm tra: 1291
Số file chứa tên bị cáo trống: 0
Tỷ lệ lỗi: 0.00%

Không có bị cáo nào bị thiếu tên trong các file.


In [8]:
def load_valid_dieus(law_doc_path):
    valid_dieus = set()
    try:
        with open(law_doc_path, 'r', encoding='utf-8') as f:
            law_data = json.load(f)
            for chuong in law_data:
                if chuong.get("type") == "Chương":
                    for dieu in chuong.get("Điều", []):
                        if dieu.get("type") == "Điều":
                            dieu_idx = str(dieu.get("index", "")).strip()
                            if dieu_idx:
                                valid_dieus.add(dieu_idx)
    except Exception as e:
        print(f"Lỗi khi load {law_doc_path}: {e}")
    return valid_dieus


valid_dieus = load_valid_dieus(LAW_DOC_PATH)
print(f"Đã load {len(valid_dieus)} Điều hợp lệ từ {LAW_DOC_PATH}.\n")

invalid_references = []
files_with_errors = set()
total_rules_checked = 0

print(f"Bắt đầu kiểm tra tính hợp lệ của các Điều luật trong thư mục '{DATA_DIR}'...")

for file_path in json_files_to_check:
    file_name = os.path.basename(file_path)
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        for dieu in set(extract_dieu_from_case(data)):
            total_rules_checked += 1
            if dieu not in valid_dieus:
                invalid_references.append({
                    "file": file_name,
                    "error": f"Điều '{dieu}' không tồn tại trong law_doc.json",
                    "dieu": dieu,
                })
                files_with_errors.add(file_name)

    except json.JSONDecodeError:
        pass
    except Exception as e:
        print(f"Lỗi khi xử lý {file_path}: {e}")

print("="*70)
print("KẾT QUẢ KIỂM TRA ĐIỀU LUẬT")
print("="*70)

if total_rules_checked == 0:
    print("Không tìm thấy trích dẫn Điều luật nào để kiểm tra.")
else:
    error_rate = (len(invalid_references) / total_rules_checked) * 100

    print(f"Tổng số file đã duyệt: {len(json_files_to_check)}")
    print(f"Tổng số Điều duy nhất theo từng file được kiểm tra: {total_rules_checked}")
    print(f"Số lượng trích dẫn không hợp lệ: {len(invalid_references)}")
    print(f"Số file chứa lỗi: {len(files_with_errors)}")
    print(f"Tỷ lệ trích dẫn lỗi: {error_rate:.2f}%\n")

    if invalid_references:
        print("CHI TIẾT CÁC LỖI TÌM THẤY:")
        for idx, err in enumerate(invalid_references[:100], 1):
            print(f"{idx}. File: {err['file']} | {err['error']}")
        if len(invalid_references) > 100:
            print(f"... và {len(invalid_references) - 100} lỗi khác.")
    else:
        print("Tất cả các Điều trích xuất được đều tồn tại trong law_doc.json.")


Đã load 427 Điều hợp lệ từ law_doc.json.

Bắt đầu kiểm tra tính hợp lệ của các Điều luật trong thư mục 'data_create/extracted_fields/2024-2025'...
KẾT QUẢ KIỂM TRA ĐIỀU LUẬT
Tổng số file đã duyệt: 1291
Tổng số Điều duy nhất theo từng file được kiểm tra: 15396
Số lượng trích dẫn không hợp lệ: 864
Số file chứa lỗi: 407
Tỷ lệ trích dẫn lỗi: 5.61%

CHI TIẾT CÁC LỖI TÌM THẤY:
1. File: 01-07-2024-Tay_Ninh-2ta1533001t1cvn.json | Điều '468' không tồn tại trong law_doc.json
2. File: 01-07-2024-Tay_Ninh-2ta1533001t1cvn.json | Điều '591' không tồn tại trong law_doc.json
3. File: 01-12-2024-Lai_Chau-2ta1704350t1cvn.json | Điều '584' không tồn tại trong law_doc.json
4. File: 02-01-2024-Binh_Duong-2ta1402551t1cvn.json | Điều '468' không tồn tại trong law_doc.json
5. File: 02-01-2024-Binh_Duong-2ta1402551t1cvn.json | Điều '584' không tồn tại trong law_doc.json
6. File: 02-01-2024-Binh_Duong-2ta1402551t1cvn.json | Điều '593' không tồn tại trong law_doc.json
7. File: 02-01-2024-Binh_Duong-2ta1402551t1c

In [ ]:
import os
import glob
import json
import re
r"""
# We condensed the 40+ phrases into 5 robust regex patterns.
# \s+ handles multiple spaces, tabs, or newlines between words.
# (?:...) creates an optional or interchangeable group without capturing it.
# Character classes like [ậaâ] catch common typing/OCR diacritic errors (e.g., kết luận vs kết luân).
REGEX_PATTERNS = [
    # 1. The Core Evaluation: (đã/có) đủ cơ sở/căn cứ/chứng cứ (để) kết luận/xác định/khẳng định
    r"(?:đã\s+)?(?:có\s+)?đủ\s+(?:c[ơo]\s*s[ởo]|c[ăa]n\s*c[ứu]|chứng\s*cứ)\s+(?:để\s+)?(?:kết\s*lu[ậaâ]n|xác\s*định|khẳng\s*định)",
    
    # 2. The Trial Panel: hội đồng xét xử / hđxx (nhận/xét) thấy
    r"(?:hội\s*đồng\s*xét\s*xử|hđxx)\s*(?:nhận|xét)?\s*thấy",
    
    # 3. The "Xét thấy" variations: xét thấy (có) đủ cơ sở/căn cứ
    r"xét\s*thấy\s*(?:có\s+)?đủ\s+(?:c[ơo]\s*s[ởo]|c[ăa]n\s*c[ứu])",
    
    # 4. Elements of crime: (đã/có) đủ yếu tố cấu thành
    r"(?:đã\s+|có\s+)?đủ\s*yếu\s*tố\s*cấu\s*thành",
    
    # 5. Context starters: xét hành vi phạm tội / xét lời khai
    r"xét\s*(?:hành\s*vi\s*phạm\s*tội|lời\s*khai)",
    
]

# Compile patterns once for faster execution across hundreds of files
COMPILED_PATTERNS = [re.compile(p, re.IGNORECASE) for p in REGEX_PATTERNS]

CHECK_FOLDER = "data_create/filled_template_openai" 

files_to_search = glob.glob(os.path.join(CHECK_FOLDER, '*.json'))

contains_any_count = 0
contains_none_count = 0

files_containing_any = []
files_containing_none = []

print("Bắt đầu tìm kiếm bằng Regex patterns...")
print(f"Trong thư mục: '{CHECK_FOLDER}'...\n")

for file_path in files_to_search:
    if file_path.endswith('law_doc.json'):
        continue
        
    file_name = os.path.basename(file_path)
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        nhan_dinh = data.get("NHAN_DINH_CUA_TOA_AN", {})
        
        # Gom tất cả văn bản trong NHAN_DINH_CUA_TOA_AN lại thành 1 chuỗi
        combined_text = ""
        if isinstance(nhan_dinh, dict):
            combined_text = " ".join(str(v) for v in nhan_dinh.values())
        elif isinstance(nhan_dinh, str): 
            combined_text = nhan_dinh
            
        # Kiểm tra xem có text nào khớp với BẤT KỲ pattern nào không
        # Chúng ta không cần .lower() nữa vì đã dùng re.IGNORECASE khi compile
        has_any = any(pattern.search(combined_text) for pattern in COMPILED_PATTERNS)
        
        if has_any:
            contains_any_count += 1
            files_containing_any.append(file_name)
        else:
            contains_none_count += 1
            files_containing_none.append(file_name)
            
    except json.JSONDecodeError:
        print(f"Lỗi đọc JSON: {file_name}")
    except Exception as e:
        print(f"Lỗi khi xử lý {file_name}: {e}")

# Display Results
print("="*75)
print(f"📊 KẾT QUẢ TÌM KIẾM BẰNG REGEX")
print("="*75)

total_searched = len([f for f in files_to_search if not f.endswith('law_doc.json')])

if total_searched == 0:
    print(f"Thư mục '{CHECK_FOLDER}' trống hoặc không tồn tại file hợp lệ.")
else:
    print(f"Tổng số file đã kiểm tra: {total_searched}")
    print(f"✅ Số file chứa ÍT NHẤT MỘT cụm từ : {contains_any_count} ({(contains_any_count/total_searched)*100:.1f}%)")
    print(f"❌ Số file KHÔNG chứa BẤT KỲ cụm từ nào: {contains_none_count} ({(contains_none_count/total_searched)*100:.1f}%)\n")
    
    if contains_none_count > 0:
        print("🛑 DANH SÁCH FILE KHÔNG KHỚP VỚI BẤT KỲ PATTERN NÀO (NONE):")
        # In ra tối đa 20 file để tránh làm trôi màn hình console
        for idx, f in enumerate(files_containing_none[:20], 1):
            print(f"  {idx}. {f}")
        if contains_none_count > 30:
            print(f"  ... và {contains_none_count - 30} file khác.")
    else:
        print("✅ Tuyệt vời! Tất cả các file đều khớp với ít nhất một pattern.")

"""

In [9]:
# Non-destructive summary of files flagged by the previous checks.
defendant_error_files = {item['file'] for item in mismatch_files}
all_bad_files = defendant_error_files.union(files_with_errors).union(empty_name_files)

total_files_scanned = len(json_files_to_check)
kept_count = total_files_scanned - len(all_bad_files)

print("="*60)
print("TỔNG HỢP FILE CẦN XEM LẠI")
print("="*60)
print(f"Thư mục nguồn: {DATA_DIR}")
print(f"Tổng số file đã quét: {total_files_scanned}")
print(f"Số file cần xem lại: {len(all_bad_files)}")
print(f"Số file không bị flag: {kept_count}\n")

if all_bad_files:
    print("Một số file cần xem lại:")
    for file_name in sorted(all_bad_files)[:100]:
        print(f"- {file_name}")
    if len(all_bad_files) > 100:
        print(f"... và {len(all_bad_files) - 100} file khác.")
else:
    print("Không có file nào bị flag bởi các bước kiểm tra hiện tại.")


TỔNG HỢP FILE CẦN XEM LẠI
Thư mục nguồn: data_create/extracted_fields/2024-2025
Tổng số file đã quét: 1291
Số file cần xem lại: 407
Số file không bị flag: 884

Một số file cần xem lại:
- 01-07-2024-Tay_Ninh-2ta1533001t1cvn.json
- 01-12-2024-Lai_Chau-2ta1704350t1cvn.json
- 02-01-2024-Binh_Duong-2ta1402551t1cvn.json
- 02-02-2024-Binh_Phuoc-2ta1436434t1cvn.json
- 02-04-2024-Ho_Chi_Minh-2ta1618592t1cvn.json
- 02-05-2024-Phu_Yen-2ta1496823t1cvn.json
- 02-07-2024-Binh_Phuoc-2ta1656003t1cvn.json
- 02-08-2024-Da_Nang-2ta1577151t1cvn.json
- 02-08-2024-Hung_Yen-2ta1591337t1cvn.json
- 03-01-2024-Binh_Phuoc-2ta1406630t1cvn.json
- 03-01-2024-Da_Nang-2ta1453331t1cvn.json
- 03-01-2024-Ha_Noi-2ta1528823t1cvn.json
- 03-05-2024-Ca_Mau-2ta1498837t1cvn.json
- 03-05-2024-Ha_Noi-2ta1626827t1cvn.json
- 03-05-2024-Ho_Chi_Minh-2ta1620215t1cvn.json
- 03-05-2024-Lai_Chau-2ta1503009t1cvn.json
- 03-05-2024-Thai_Binh-2ta1510909t1cvn.json
- 03-05-2024-Thai_Nguyen-2ta1615305t1cvn.json
- 03-06-2024-Lang_Son-2ta1545213

In [10]:
duplicate_defendant_files = []
total_files = 0

print(f"Bắt đầu kiểm tra bị cáo lặp lại trong 'Danh_sach_bi_cao' tại thư mục '{DATA_DIR}'...\n")

for file_path in json_files_to_check:
    file_name = os.path.basename(file_path)

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        names_by_source = get_defendant_names(data)
        defendants = sorted(names_by_source.get("Danh_sach_bi_cao", set()))
        name_counts = Counter(defendants)
        duplicates = {name: count for name, count in name_counts.items() if count > 1}

        if duplicates:
            duplicate_defendant_files.append({
                "file": file_name,
                "duplicates": duplicates,
            })

        total_files += 1

    except json.JSONDecodeError:
        print(f"Lỗi đọc JSON: {file_name}")
    except Exception as e:
        print(f"Lỗi khi xử lý {file_name}: {e}")

print("="*75)
print("KẾT QUẢ KIỂM TRA BỊ CÁO LẶP LẠI")
print("="*75)

if total_files == 0:
    print("Không tìm thấy file JSON nào hợp lệ trong thư mục để kiểm tra.")
else:
    dup_count = len(duplicate_defendant_files)
    dup_rate = (dup_count / total_files) * 100

    print(f"Tổng số bản án đã kiểm tra: {total_files}")
    print(f"Số bản án có bị cáo lặp lại: {dup_count} ({dup_rate:.2f}%)")

    if dup_count > 0:
        print("\nCHI TIẾT CÁC BẢN ÁN BỊ LẶP BỊ CÁO:")
        for item in duplicate_defendant_files[:100]:
            print(f"- File: {item['file']}")
            for name, count in item['duplicates'].items():
                print(f"  + Bị cáo '{name}': xuất hiện {count} lần")
        if dup_count > 100:
            print(f"... và {dup_count - 100} file khác.")
    else:
        print("\nKhông có bản án nào chứa bị cáo bị lặp lại trong danh sách bị cáo.")


Bắt đầu kiểm tra bị cáo lặp lại trong 'Danh_sach_bi_cao' tại thư mục 'data_create/extracted_fields/2024-2025'...

KẾT QUẢ KIỂM TRA BỊ CÁO LẶP LẠI
Tổng số bản án đã kiểm tra: 1291
Số bản án có bị cáo lặp lại: 0 (0.00%)

Không có bản án nào chứa bị cáo bị lặp lại trong danh sách bị cáo.


In [ ]:
import os
import glob
import json
import re
r"""
# 1. Compile regex patterns for the Summary chunk
REGEX_PATTERNS = [
    r"(?:đã\s+)?(?:có\s+)?đủ\s+(?:c[ơo]\s*s[ởo]|c[ăa]n\s*c[ứu]|chứng\s*cứ)\s+(?:để\s+)?(?:kết\s*lu[ậaâ]n|xác\s*định|khẳng\s*định)",
    r"(?:hội\s*đồng\s*xét\s*xử|hđxx)\s*(?:nhận|xét)?\s*thấy",
    r"xét\s*thấy\s*(?:có\s+)?đủ\s+(?:c[ơo]\s*s[ởo]|c[ăa]n\s*c[ứu])",
    r"(?:đã\s+|có\s+)?đủ\s*yếu\s*tố\s*cấu\s*thành",
    r"xét\s*(?:hành\s*vi\s*phạm\s*tội|lời\s*khai)"
]
COMPILED_PATTERNS = [re.compile(p, re.IGNORECASE) for p in REGEX_PATTERNS]

# 2. Setup directory (update files in place)
CHECK_FOLDER = "data_create/filled_template_openai"
json_files = glob.glob(os.path.join(CHECK_FOLDER, '*.json'))

print(f"Bắt đầu append Summary/Tang_nang/Giam_nhe vào file gốc cho {len(json_files)} file...")

processed_count = 0

for file_path in json_files:
    if file_path.endswith('law_doc.json'):
        continue

    file_name = os.path.basename(file_path)

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        nhan_dinh = data.get("NHAN_DINH_CUA_TOA_AN", {})

        # --- Extract text chunks from NHAN_DINH ---
        summary_blocks = []
        tang_nang_blocks = []
        giam_nhe_blocks = []

        if isinstance(nhan_dinh, dict):
            for key, text in nhan_dinh.items():
                if str(key).strip() == "[1]":
                    continue

                text_str = str(text)
                text_lower = text_str.lower()

                if any(pattern.search(text_str) for pattern in COMPILED_PATTERNS):
                    summary_blocks.append(text_str)

                if "tình tiết tăng nặng" in text_lower:
                    tang_nang_blocks.append(text_str)

                if "tình tiết giảm nhẹ" in text_lower:
                    giam_nhe_blocks.append(text_str)

        # Append only requested blocks into the original JSON object
        data["Summary"] = "\n\n".join(summary_blocks)
        data["Tang_nang"] = "\n\n".join(tang_nang_blocks)
        data["Giam_nhe"] = "\n\n".join(giam_nhe_blocks)

        with open(file_path, 'w', encoding='utf-8') as out_f:
            json.dump(data, out_f, ensure_ascii=False, indent=4)

        processed_count += 1

    except json.JSONDecodeError:
        print(f"Lỗi đọc JSON: {file_name}")
    except Exception as e:
        print(f"Lỗi khi xử lý {file_name}: {e}")

print("=" * 70)
print(f"✅ Đã hoàn thành append dữ liệu vào {processed_count} file gốc (không tạo folder/file mới).")
print(f"📂 File được cập nhật trực tiếp trong thư mục: '{CHECK_FOLDER}'")
print("=" * 70)
"""

In [ ]:
# %%
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    PLOTTING_AVAILABLE = True
except ModuleNotFoundError as e:
    PLOTTING_AVAILABLE = False
    print(f"Bỏ qua biểu đồ vì thiếu thư viện: {e.name}")

# Có thể đổi Chương cần phân tích tại đây.
TARGET_CHAPTER = globals().get("TARGET_CHAPTER", "XXII")

chuong_dieus = []
try:
    with open(LAW_DOC_PATH, 'r', encoding='utf-8') as f:
        law_data = json.load(f)
        for chuong in law_data:
            if str(chuong.get("index", "")) == TARGET_CHAPTER:
                for dieu in chuong.get("Điều", []):
                    dieu_idx = str(dieu.get("index", "")).strip()
                    if dieu_idx:
                        chuong_dieus.append(dieu_idx)
                break
except Exception as e:
    print(f"Lỗi khi load {LAW_DOC_PATH}: {e}")

if not chuong_dieus:
    print(f"Không tìm thấy Điều nào cho Chương {TARGET_CHAPTER} trong law_doc.json")
else:
    print(f"Chương {TARGET_CHAPTER} kéo dài từ Điều {chuong_dieus[0]} đến Điều {chuong_dieus[-1]} (Tổng: {len(chuong_dieus)} Điều).")

    case_files_target = []
    dieu_counts = {d: 0 for d in chuong_dieus}

    for file_path in case_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            extracted = set(extract_dieu_from_case(data))
            if any(dieu_to_chuong.get(d) == TARGET_CHAPTER for d in extracted):
                case_files_target.append(file_path)

            for d in extracted:
                if d in dieu_counts:
                    dieu_counts[d] += 1
        except Exception:
            pass

    print(f"Số file thuộc Chương {TARGET_CHAPTER}: {len(case_files_target)}")

    if PLOTTING_AVAILABLE:
        plt.figure(figsize=(18, 6))
        x_labels = chuong_dieus
        y_values = [dieu_counts[d] for d in x_labels]

        sns.barplot(x=x_labels, y=y_values, color="steelblue")
        plt.title(f"Tần suất vi phạm các Điều trong Chương {TARGET_CHAPTER}", fontsize=16, pad=15)
        plt.xlabel(f"Các Điều (Từ {chuong_dieus[0]} đến Điều {chuong_dieus[-1]})", fontsize=12)
        plt.ylabel("Số lượng vụ án", fontsize=12)
        plt.xticks(rotation=90, fontsize=9)
        plt.tight_layout()
        plt.show()


In [ ]:
try:
    import pandas as pd
    import numpy as np
    CORRELATION_DEPS_AVAILABLE = True
except ModuleNotFoundError as e:
    CORRELATION_DEPS_AVAILABLE = False
    print(f"Bỏ qua ma trận tương quan vì thiếu thư viện: {e.name}")

if not CORRELATION_DEPS_AVAILABLE:
    pass
elif not chuong_dieus:
    print("Không có dữ liệu Điều để tính tương quan.")
else:
    occurrence_data = []

    for file_path in case_files_target:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            extracted = set(extract_dieu_from_case(data))
            row = {d: (1 if d in extracted else 0) for d in chuong_dieus}
            occurrence_data.append(row)
        except Exception:
            pass

    df_occurrences = pd.DataFrame(occurrence_data)

    if df_occurrences.empty:
        print(f"Không có vụ án nào thuộc Chương {TARGET_CHAPTER} để tính tương quan.")
    else:
        df_filtered = df_occurrences.loc[:, df_occurrences.sum() > 0]

        if df_filtered.shape[1] < 2:
            print("Không đủ số lượng Điều bị vi phạm (cần ít nhất 2 Điều khác nhau) để tính toán ma trận tương quan.")
        else:
            corr_matrix = df_filtered.corr()

            if PLOTTING_AVAILABLE:
                plt.figure(figsize=(14, 10))
                mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

                sns.heatmap(
                    corr_matrix,
                    mask=mask,
                    cmap="coolwarm",
                    vmax=1,
                    vmin=-1,
                    center=0,
                    square=True,
                    linewidths=.5,
                    cbar_kws={"shrink": .75},
                    annot=True,
                    fmt=".2f",
                    annot_kws={"size": 8},
                )

                plt.title(
                    f"Tương quan vi phạm đồng thời giữa các Điều - Chương {TARGET_CHAPTER}\n"
                    "(Chỉ hiển thị các Điều có dữ liệu)",
                    fontsize=15,
                    pad=15,
                )
                plt.tight_layout()
                plt.show()
            else:
                print("Đã tính corr_matrix, nhưng bỏ qua heatmap vì thiếu matplotlib/seaborn.")
